# HumAID — Zero-shot Classification (Filtered Labels, Batch API, Sharding)

- **Filtered labels (per event):** prompts + JSON schema only list labels that appear in that event’s ground truth → reduces out-of-scope (OOS) predictions.
- **Batch API flow:** build `requests.jsonl` → upload → create batch → poll → download `outputs.jsonl` (and `errors.jsonl` if any).
- **Patch pass:** after batch completes, any missing/blank predictions are re-classified synchronously so `predictions.csv` has one row per input.
- **Stratified sharding (optional):** split large events into *k* shards **preserving class ratios**; use the **same** event-level labels + rules for all shards; merge predictions back in original order.
- **Reporting:** confusion matrices (counts + row-normalized), per-class F1/error, mistakes CSV, and a sortable `results/index.html`.  
  - **Scope** = label universe used for metrics (default `truth`).  
  - **OOS preds** = predictions not in the truth set (QA signal).

## Key settings
- `MODEL` (e.g., `gpt-4o`), `RULES` (e.g., `RULES_1`), `TAG`
- `DRYRUN_N`, `POLL_SECS`
- Token budgeting: `BATCH_TOKEN_LIMIT`, `SAFETY_MARGIN`, `MAX_OUTPUT_TOKENS`
- `.env` with `OPENAI_API_KEY_1` (and optionally a second key)

# 0) Setup

In [1]:
from pathlib import Path
import math
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment
from humaidclf import build_token_index               # token budgeting (sampling-based)
from humaidclf import run_experiment_sharded          # NEW: stratified sharded runner
from humaidclf.batch import use_api_key_env           # (optional) keep key switcher
from rules import RULES_1

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS = ["train"]             # or ["train","dev","test"]
MODEL = "gpt-4o"
RULES = RULES_1
TAG = "modeS-gpt-4o-RULES1-filtered"
DRYRUN_N = 20
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"

# Token caps & estimates
BATCH_TOKEN_LIMIT = 5_000_000   # Tier-1 (4o): 90,000, Tier-3 (4o): 5,000,000
SAFETY_MARGIN = 0.90            # use only 90% of the cap
MAX_OUTPUT_TOKENS = 40          # matches your request schema

# 1) Discover datasets (events/splits)

In [2]:
def discover_tsvs(base: Path, splits: list[str]):
    items = []
    for event_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        event = event_dir.name
        for split in splits:
            tsv = event_dir / f"{event}_{split}.tsv"
            if tsv.exists():
                items.append({"event": event, "split": split, "tsv": str(tsv)})
    return pd.DataFrame(items)

df_sources = discover_tsvs(BASE, SPLITS)

# --- token budgeting ---
token_index = build_token_index(
    df_sources,
    model=MODEL,
    rules_text=RULES,
    batch_token_limit=BATCH_TOKEN_LIMIT,
    safety_margin=SAFETY_MARGIN,
    sample_size=200,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)

display(token_index)

df_fit     = token_index[token_index["fits_cap"]].reset_index(drop=True)
df_too_big = token_index[~token_index["fits_cap"]].reset_index(drop=True)

print("OK to run as single batch:")
display(df_fit[["event","split","num_rows","est_total_tokens","limit_used_%"]])

print("Will be sharded (exceeds cap):")
display(df_too_big[["event","split","num_rows","est_total_tokens","limit_used_%"]])

,event,split,tsv,num_rows,avg_req_tokens,est_total_tokens,fits_cap,limit_used_%
1,canada_wildfires_2016,train,Dataset\HumAID\canada_wildfires_2016\canada_wi...,1569,473,742137,False,824.6
8,kaikoura_earthquake_2016,train,Dataset\HumAID\kaikoura_earthquake_2016\kaikou...,1536,486,746496,False,829.4
2,cyclone_idai_2019,train,Dataset\HumAID\cyclone_idai_2019\cyclone_idai_...,2753,518,1426054,False,1584.5
4,hurricane_florence_2018,train,Dataset\HumAID\hurricane_florence_2018\hurrica...,4384,501,2196384,False,2440.4
7,hurricane_maria_2017,train,Dataset\HumAID\hurricane_maria_2017\hurricane_...,5094,488,2485872,False,2762.1
0,california_wildfires_2018,train,Dataset\HumAID\california_wildfires_2018\calif...,5163,508,2622804,False,2914.2
3,hurricane_dorian_2019,train,Dataset\HumAID\hurricane_dorian_2019\hurricane...,5329,501,2669829,False,2966.5
9,kerala_floods_2018,train,Dataset\HumAID\kerala_floods_2018\kerala_flood...,5588,506,2827528,False,3141.7
5,hurricane_harvey_2017,train,Dataset\HumAID\hurricane_harvey_2017\hurricane...,6378,486,3099708,False,3444.1
6,hurricane_irma_2017,train,Dataset\HumAID\hurricane_irma_2017\hurricane_i...,6579,486,3197394,False,3552.7


OK to run as single batch:


,event,split,num_rows,est_total_tokens,limit_used_%


Will be sharded (exceeds cap):


,event,split,num_rows,est_total_tokens,limit_used_%
0,canada_wildfires_2016,train,1569,742137,824.6
1,kaikoura_earthquake_2016,train,1536,746496,829.4
2,cyclone_idai_2019,train,2753,1426054,1584.5
3,hurricane_florence_2018,train,4384,2196384,2440.4
4,hurricane_maria_2017,train,5094,2485872,2762.1
5,california_wildfires_2018,train,5163,2622804,2914.2
6,hurricane_dorian_2019,train,5329,2669829,2966.5
7,kerala_floods_2018,train,5588,2827528,3141.7
8,hurricane_harvey_2017,train,6378,3099708,3444.1
9,hurricane_irma_2017,train,6579,3197394,3552.7


# 2) Run all datasets (sequentially)

In [3]:
def run_list_single(dflist: pd.DataFrame, rules_text: str, model: str, tag: str):
    """Run events that already fit under the cap using the normal runner."""
    results = []
    for _, row in dflist.iterrows():
        event, split, tsv = row["event"], row["split"], row["tsv"]
        print(f"\n=== Running (single) {event}/{split} ({model} | {tag}) ===")
        try:
            plan, preds, summary = run_experiment(
                dataset_path=tsv,
                rules=rules_text,
                model=model,
                tag=tag,
                dryrun_n=DRYRUN_N,
                poll_secs=POLL_SECS,
                out_root=OUT_ROOT,
                do_analysis=DO_ANALYSIS,
            )
            acc = summary.get("accuracy") if summary else float("nan")
            f1  = summary.get("macro_f1") if summary else float("nan")
            n   = summary.get("num_total_with_truth") if summary else len(preds)
            results.append({
                "event": event, "split": split,
                "run_dir": str(plan["dir"]),
                "predictions_csv": str(plan["predictions_csv"]),
                "macro_f1": f1, "accuracy": acc, "num_total": n,
                "mode": "single",
            })
        except Exception as e:
            print(f"[ERROR] {event}/{split}: {e}")
            results.append({
                "event": event, "split": split, "run_dir": "ERROR",
                "predictions_csv": "", "macro_f1": float("nan"),
                "accuracy": float("nan"), "num_total": 0, "mode": "single",
            })
    return pd.DataFrame(results)

def run_list_sharded(dflist: pd.DataFrame, token_df: pd.DataFrame, rules_text: str, model: str, tag: str):
    """Run events that exceed the cap using stratified shards. k is computed from token estimates."""
    results = []
    # Build a quick lookup: (event,split) -> est_total_tokens
    est_map = {(r.event, r.split): r.est_total_tokens for r in token_df.itertuples(index=False)}
    eff_cap = BATCH_TOKEN_LIMIT * SAFETY_MARGIN

    for _, row in dflist.iterrows():
        event, split, tsv = row["event"], row["split"], row["tsv"]
        est_tokens = est_map.get((event, split), None)
        # Conservative shard count: ceil(est / eff_cap). Min 2.
        k = max(2, math.ceil((est_tokens or (eff_cap + 1)) / eff_cap))
        print(f"\n=== Running (sharded x{k}) {event}/{split} ({model} | {tag}) ===")
        try:
            plan, preds, summary = run_experiment_sharded(
                dataset_path=tsv,
                rules=rules_text,
                model=model,
                tag=f"{tag}-sharded{k}",
                k_shards=k,
                temperature=0.0,
                poll_secs=POLL_SECS,
                out_root=OUT_ROOT,
                do_analysis=DO_ANALYSIS,
                analysis_subdir="analysis",  # merged analysis
            )
            acc = summary.get("accuracy") if summary else float("nan")
            f1  = summary.get("macro_f1") if summary else float("nan")
            n   = summary.get("num_total_with_truth") if summary else len(preds)
            results.append({
                "event": event, "split": split,
                "run_dir": str(plan["dir"]),
                "predictions_csv": str(plan["predictions_csv"]),
                "macro_f1": f1, "accuracy": acc, "num_total": n,
                "mode": f"sharded{k}",
            })
        except Exception as e:
            print(f"[ERROR] {event}/{split}: {e}")
            results.append({
                "event": event, "split": split, "run_dir": "ERROR",
                "predictions_csv": "", "macro_f1": float("nan"),
                "accuracy": float("nan"), "num_total": 0, "mode": f"sharded{k}",
            })
    return pd.DataFrame(results)

In [4]:
# --- Run singles with your normal key (optional context manager kept for parity)
with use_api_key_env("OPENAI_API_KEY_1"):
    print(">>> Using OPENAI_API_KEY_1")
    df_runs_single = run_list_single(df_fit, RULES, MODEL, tag=f"{TAG}-TIER1")

# --- Run sharded for the too-big ones (same key or another if you prefer)
# You can keep the same key; sharding is already controlling token usage.
with use_api_key_env("OPENAI_API_KEY_1"):
    if not df_too_big.empty:
        df_runs_sharded = run_list_sharded(df_too_big, token_index, RULES, MODEL, tag=f"{TAG}")
    else:
        df_runs_sharded = pd.DataFrame()
        print("No large datasets to shard.")

# 3) Save a small index of all runs
from datetime import datetime
idx_dir = Path(OUT_ROOT) / "_indexes"
idx_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")

all_runs = pd.concat([df_runs_single, df_runs_sharded], ignore_index=True)
all_runs.to_csv(idx_dir / f"runs_{MODEL}_{TAG}_{stamp}.csv", index=False)
print("Saved run index at:", idx_dir)
display(all_runs)

>>> Using OPENAI_API_KEY_1

=== Running (sharded x10) canada_wildfires_2016/train (gpt-4o | modeS-gpt-4o-RULES1-filtered) ===
[batch batch_6918342c9b0c8190aa7b6c46b20358b0] status = validating
[batch batch_6918342c9b0c8190aa7b6c46b20358b0] status = completed
[batch batch_6918355c33fc8190bf903cfff4170721] status = validating
[batch batch_6918355c33fc8190bf903cfff4170721] status = in_progress
[batch batch_6918355c33fc8190bf903cfff4170721] status = in_progress
[batch batch_6918355c33fc8190bf903cfff4170721] status = in_progress
[batch batch_6918355c33fc8190bf903cfff4170721] status = completed
[batch batch_69183a0ff0248190acaf96fe9f87b4ca] status = validating
[batch batch_69183a0ff0248190acaf96fe9f87b4ca] status = in_progress
[batch batch_69183a0ff0248190acaf96fe9f87b4ca] status = in_progress
[batch batch_69183a0ff0248190acaf96fe9f87b4ca] status = in_progress
[batch batch_69183a0ff0248190acaf96fe9f87b4ca] status = completed
[batch batch_69183ec43994819087dd89742c82f3f5] status = validating


,event,split,run_dir,predictions_csv,macro_f1,accuracy,num_total,mode
0,canada_wildfires_2016,train,runs\canada_wildfires_2016\train\gpt-4o\202511...,runs\canada_wildfires_2016\train\gpt-4o\202511...,0.658426,0.789038,1569,sharded10
1,kaikoura_earthquake_2016,train,runs\kaikoura_earthquake_2016\train\gpt-4o\202...,runs\kaikoura_earthquake_2016\train\gpt-4o\202...,0.701267,0.718099,1536,sharded10
2,cyclone_idai_2019,train,ERROR,,NaN,NaN,0,sharded18
3,hurricane_florence_2018,train,runs\hurricane_florence_2018\train\gpt-4o\2025...,runs\hurricane_florence_2018\train\gpt-4o\2025...,0.667990,0.744754,4384,sharded28
4,hurricane_maria_2017,train,runs\hurricane_maria_2017\train\gpt-4o\2025111...,runs\hurricane_maria_2017\train\gpt-4o\2025111...,0.641114,0.662740,5094,sharded31
5,california_wildfires_2018,train,runs\california_wildfires_2018\train\gpt-4o\20...,runs\california_wildfires_2018\train\gpt-4o\20...,0.633117,0.719349,5163,sharded33
6,hurricane_dorian_2019,train,runs\hurricane_dorian_2019\train\gpt-4o\202511...,runs\hurricane_dorian_2019\train\gpt-4o\202511...,0.584196,0.628636,5329,sharded33
7,kerala_floods_2018,train,ERROR,,NaN,NaN,0,sharded35
8,hurricane_harvey_2017,train,runs\hurricane_harvey_2017\train\gpt-4o\202511...,runs\hurricane_harvey_2017\train\gpt-4o\202511...,0.619240,0.652556,6378,sharded39
9,hurricane_irma_2017,train,ERROR,,NaN,NaN,0,sharded40


# Re-run the ERROR entries manually

In [23]:
copy_events = [
    "cyclone_idai_2019",
    "kerala_floods_2018",
    "hurricane_irma_2017"
]    
df_rerun = df_too_big[df_too_big["event"].isin(copy_events)].copy()
display(df_rerun)
# df_errors = df_runs_sharded[df_too_big["run_dir"] == "ERROR"].copy()
# display(df_errors)


,event,split,tsv,num_rows,avg_req_tokens,est_total_tokens,fits_cap,limit_used_%
2,cyclone_idai_2019,train,Dataset\HumAID\cyclone_idai_2019\cyclone_idai_...,2753,518,1426054,False,1584.5
7,kerala_floods_2018,train,Dataset\HumAID\kerala_floods_2018\kerala_flood...,5588,506,2827528,False,3141.7
9,hurricane_irma_2017,train,Dataset\HumAID\hurricane_irma_2017\hurricane_i...,6579,486,3197394,False,3552.7


In [24]:
# --- Run singles with your normal key (optional context manager kept for parity)
with use_api_key_env("OPENAI_API_KEY_1"):
    print(">>> Using OPENAI_API_KEY_1")
    df_runs_single = run_list_single(df_rerun, RULES, MODEL, tag=f"{TAG}-TIER1")

# placeholder since we don't have sharded experiments
df_runs_sharded = pd.DataFrame()

# Save a small index of all runs
from datetime import datetime
idx_dir = Path(OUT_ROOT) / "_indexes"
idx_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")

all_runs = pd.concat([df_runs_single, df_runs_sharded], ignore_index=True)
all_runs.to_csv(idx_dir / f"runs_{MODEL}_{TAG}_{stamp}.csv", index=False)
print("Saved run index at:", idx_dir)
display(all_runs)

>>> Using OPENAI_API_KEY_1

=== Running (single) cyclone_idai_2019/train (gpt-4o | modeS-gpt-4o-RULES1-filtered-TIER1) ===
Macro-F1 (tiny sample): 0.5962962962962963
[batch batch_691a39ab708c819089deeeff217a9793] status = validating
[batch batch_691a39ab708c819089deeeff217a9793] status = in_progress
[batch batch_691a39ab708c819089deeeff217a9793] status = in_progress
[batch batch_691a39ab708c819089deeeff217a9793] status = in_progress
[batch batch_691a39ab708c819089deeeff217a9793] status = completed
[batch batch_691a39ab708c819089deeeff217a9793] final status = completed
Saved predictions to: runs\cyclone_idai_2019\train\gpt-4o\20251116-125253-modeS-gpt-4o-RULES1-filtered-TIER1\predictions.csv
Macro-F1: 0.5879460983659375

=== Running (single) kerala_floods_2018/train (gpt-4o | modeS-gpt-4o-RULES1-filtered-TIER1) ===
Macro-F1 (tiny sample): 0.6813664596273291
[batch batch_691a3e7569b88190aa11a60fc8958d74] status = validating
[batch batch_691a3e7569b88190aa11a60fc8958d74] status = in_progr

,event,split,run_dir,predictions_csv,macro_f1,accuracy,num_total,mode
0,cyclone_idai_2019,train,runs\cyclone_idai_2019\train\gpt-4o\20251116-1...,runs\cyclone_idai_2019\train\gpt-4o\20251116-1...,0.587946,0.742463,2753,single
1,kerala_floods_2018,train,runs\kerala_floods_2018\train\gpt-4o\20251116-...,runs\kerala_floods_2018\train\gpt-4o\20251116-...,0.581271,0.704009,5588,single
2,hurricane_irma_2017,train,runs\hurricane_irma_2017\train\gpt-4o\20251116...,runs\hurricane_irma_2017\train\gpt-4o\20251116...,0.605062,0.618787,6579,single


# Other experiments

In [ ]:
from pathlib import Path
import math
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment
from humaidclf import build_token_index               # token budgeting (sampling-based)
from humaidclf import run_experiment_sharded          # NEW: stratified sharded runner
from humaidclf.batch import use_api_key_env           # (optional) keep key switcher
from rules import RULES_1

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS = ["train"]             # or ["train","dev","test"]
MODEL = "gpt-4o"
RULES = RULES_1
TAG = "modeS-gpt-4o-RULES1-filtered"
DRYRUN_N = 20
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"

K = 1  # number of stratified shards

with use_api_key_env("OPENAI_API_KEY"):
    plan, preds, summary = run_experiment_sharded(
        dataset_path=str(BASE / "kerala_floods_2018" / "kerala_floods_2018_train.tsv"),
        rules=RULES,
        model=MODEL,
        tag=f"{TAG}-sharded{K}",
        k_shards=K,
        temperature=0.0,
        poll_secs=POLL_SECS,
        out_root=OUT_ROOT,
        do_analysis=DO_ANALYSIS,
        analysis_subdir="analysis",
    )

summary



In [ ]:
from pathlib import Path
import math
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment
from humaidclf import build_token_index               # token budgeting (sampling-based)
from humaidclf import run_experiment_sharded          # NEW: stratified sharded runner
from humaidclf.batch import use_api_key_env           # (optional) keep key switcher
from rules import RULES_1

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS = ["train"]             # or ["train","dev","test"]
MODEL = "gpt-4o"
RULES = RULES_1
TAG = "modeS-gpt-4o-RULES1-filtered"
DRYRUN_N = 20
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"

K = 1  # number of stratified shards

with use_api_key_env("OPENAI_API_KEY"):
    plan, preds, summary = run_experiment_sharded(
        dataset_path=str(BASE / "cyclone_idai_2019" / "cyclone_idai_2019_train.tsv"),
        rules=RULES,
        model=MODEL,
        tag=f"{TAG}-sharded{K}",
        k_shards=K,
        temperature=0.0,
        poll_secs=POLL_SECS,
        out_root=OUT_ROOT,
        do_analysis=DO_ANALYSIS,
        analysis_subdir="analysis",
    )

summary